In [19]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import pandas as pd

In [36]:
df_pre = pd.read_parquet("../output_bkp/13-21", engine='pyarrow')
df_post = pd.read_parquet("../output_bkp/22-26", engine='pyarrow')

In [37]:
print(f"Fecha Mínima en el dataset: {df_post['FECHA'].min()}")
print(f"Fecha Máxima en el dataset: {df_post['FECHA'].max()}")

Fecha Mínima en el dataset: 2022-01-01 00:00:00
Fecha Máxima en el dataset: 2026-06-30 00:00:00


In [38]:
df_pre['ANIO'] = df_pre['FECHA'].dt.year
df_pre['MES'] = df_pre['FECHA'].dt.month
df_pre['DIA'] = df_pre['FECHA'].dt.day
df_pre['NOMBRE_DIA'] = df_pre['FECHA'].dt.day_name(locale='es_ES')

MemoryError: Unable to allocate 686. MiB for an array with shape (89967310,) and data type uint64

In [23]:
orden_dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']

df_pre['NOMBRE_DIA'] = pd.Categorical(df_pre['NOMBRE_DIA'], categories=orden_dias, ordered=True)

In [24]:
df_pre.head(5)

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL,ANIO,MES,DIA,NOMBRE_DIA
0,2013-06-01,05:15,05:29,5,D,9 DE JULIO,9JULIO_S,TURN01,LINEA_D_9JULIO_S_TURN01,2,2013,6,1,Sábado
1,2013-06-01,05:30,05:44,5,D,9 DE JULIO,9JULIO_N,TURN02,LINEA_D_9JULIO_N_TURN02,1,2013,6,1,Sábado
2,2013-06-01,05:30,05:44,5,D,9 DE JULIO,9JULIO_S,TURN01,LINEA_D_9JULIO_S_TURN01,1,2013,6,1,Sábado
3,2013-06-01,05:30,05:44,5,D,9 DE JULIO,9JULIO_S,TURN02,LINEA_D_9JULIO_S_TURN02,1,2013,6,1,Sábado
4,2013-06-01,05:30,05:44,5,D,9 DE JULIO,9JULIO_S,TURN03,LINEA_D_9JULIO_S_TURN03,1,2013,6,1,Sábado


In [25]:
df_grpd_dia = df_pre.groupby(['NOMBRE_DIA', 'LINEA'])['PAX_TOTAL'].sum().reset_index()

C:\Users\Cea\AppData\Local\Temp\ipykernel_21140\3801017419.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_grpd_dia = df_pre.groupby(['NOMBRE_DIA', 'LINEA'])['PAX_TOTAL'].sum().reset_index()


# Gráficos de Línea

In [26]:
fig = px.line(df_grpd_dia, x='NOMBRE_DIA', y='PAX_TOTAL', color='LINEA', category_orders={'NOMBRE_DIA': orden_dias}, title='Pasajeros por Día de la Semana y Línea')
fig.show()

In [27]:
df_grpd_anio = df_pre.groupby(['ANIO', 'LINEA'])['PAX_TOTAL'].sum().reset_index()
fig = px.line(df_grpd_anio, x='ANIO', y='PAX_TOTAL', color='LINEA', category_orders={'NOMBRE_DIA': orden_dias}, title='Pasajeros por Año')
fig.show()

In [28]:
df_grpd_hora = df_pre.groupby(['HORA', 'LINEA'])['PAX_TOTAL'].sum().reset_index()
fig = px.line(df_grpd_hora, x='HORA', y='PAX_TOTAL', color='LINEA', category_orders={'NOMBRE_DIA': orden_dias}, title='Pasajeros por Hora y Línea')
fig.show()

# Gráficos de Barras

In [29]:
df_grpd_linea = df_pre.groupby(['LINEA'])['PAX_TOTAL'].sum().reset_index()
fig = px.bar(df_grpd_linea, x='LINEA', y='PAX_TOTAL', text_auto=True,title='Pasajeros por Línea')
fig.show()

In [30]:
fig = px.bar(df_grpd_anio, x='ANIO', y='PAX_TOTAL', color='LINEA', title='Pasajeros por Línea')
fig.show()

# Scatter Plots

In [31]:
df_grpd_hora = df_pre.groupby(['HORA'])['PAX_TOTAL'].sum().reset_index()
fig = px.scatter(df_grpd_hora, x='HORA', y='PAX_TOTAL', title='Pasajeros por Hora')
fig.show()

# Boxplot

In [32]:
df_sample = df_pre.sample(n=10000, random_state=42) # toma una muestra de 10,000 filas para el boxplot

In [33]:
df_sample = df_pre.groupby("HORA", group_keys=False).sample(frac=0.01, random_state=42)


fig = px.box(df_sample, x="HORA", y="PAX_TOTAL",
             title="Distribución de pasajeros por hora (muestra)", points=False)
fig.show()

# Heatmaps

In [34]:
df_sample = df_pre.groupby(["HORA"], group_keys=False, observed=True).sample(frac=0.001, random_state=42)

fig = px.density_heatmap(df_sample, x="HORA", y="NOMBRE_DIA", z="PAX_TOTAL", 
                         histfunc="sum", 
                         title="Mapa de calor de pasajeros por hora y día de la semana (muestra)", 
                         nbinsx=34, nbinsy=34, 
                         color_continuous_scale="Viridis", 
                         category_orders={"NOMBRE_DIA": orden_dias})
fig.show()

Para que el análisis sea más útil, habría que desagregar por línea y por estación.

In [35]:
df_sample = df_pre.groupby(["HORA"], group_keys=False, observed=True).sample(frac=0.001, random_state=42)

fig = px.density_heatmap(df_sample, x="HORA", y="LINEA", z="PAX_TOTAL", 
                         histfunc="sum", 
                         title="Mapa de calor de pasajeros por hora y línea (muestra)", 
                         nbinsx=34, nbinsy=34, 
                         color_continuous_scale="Viridis", 
                         category_orders={"NOMBRE_DIA": orden_dias})
fig.show()